# Run SOM with different configurations

- 1. Independent HFR 
    - a) Solo Coffs Harbour
    - b) Solo Newcastle 

- 2. Solo HFR + Temperature or Chl

- 3. Both HFR together

## Data & Packages

In [51]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? y


In [52]:
#------------------------------------------------------
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xpysom import XPySom
from datetime import datetime
#------------------------------------------------------

git = '/media/nbourg/One Touch/PhD/Separation_HFR_EAC/'

path_data = git+'Data/'
coffs = xr.open_dataset(path_data+'HFR/Vectors_Filled/currents_dineof_COFFS_Y2019M07_Y2020M12.nc')
newc = xr.open_dataset(path_data+'HFR/Vectors_Filled/currents_dineof_NEWC_Y2019M07_Y2021M12.nc')

path_save_soms = git+'Analysis/Self_Organizing_Maps/SOM_Different_Tests/saved_soms/'

## Functions

In [53]:
def run_som_only_radar(u,v,rows,cols,name):

    # choose the analysis you want
    som_space = True
    som_time = False

    #%% Reshape and normalize the dataset 
    [s_t, s_x, s_y] = u.shape
    u_r = np.reshape(u, (s_t, s_x*s_y))
    v_r = np.reshape(v, (s_t, s_x*s_y))

    variables_normed = [u_r, v_r]
    variables = [u_r, v_r]
    store_mean = np.zeros(len(variables))
    store_std = np.zeros(len(variables))

    #%% Reshape and normalize the dataset 

    [s_t, s_x, s_y] = u.shape
    u_r = np.reshape(u, (s_t, s_x*s_y))
    v_r = np.reshape(v, (s_t, s_x*s_y))
    variables_normed = [u_r, v_r]
    variables = [u_r, v_r]
    store_mean = np.zeros(len(variables))
    store_std = np.zeros(len(variables))

    for i in range(0,len(variables_normed)):
        variables_normed[i] =  (variables[i] - np.nanmean(variables[i])) / np.nanstd(variables[i])
        store_mean[i] = np.nanmean(variables[i])
        store_std[i] = np.nanstd(variables[i])

    # Normalizing
    if som_space:
        data_nan = np.concatenate(variables_normed[:],
                                  axis=1)
        print('You have chosen SOM spatial analysis')
        print('Your data shape is '+str(data_nan.shape))

    #Here the dataset is completely full 
    mask_nan = np.isnan(data_nan[0,:])
    data = data_nan[:,~mask_nan]


    #%% Run SOM 
    data[np.isnan(data)]=0

    """
    For spatial SOM: time(rows) * variables(columns), 
    variables are the concatenation of U and V values for each pixel 

    For temporal SOM: pixels(rows) * variables(columns), 
    variables are the concatenation of U and V values for each timestep 

    """

    number_neurons = rows * cols;

    n_init = 200 # number of iterations in the training
    sigma = 1.5 # radius of the neighborhood function
    learning_rate = 1

    if som_time:
        data_run = data.T.copy()
    elif som_space:
        data_run = data.copy()

    start_time = datetime.now()

    som = XPySom(cols, rows, data_run.shape[1],
                sigma = sigma, learning_rate = learning_rate,
                topology = 'hexagonal', xp = np)

    print('initializing')
    som.pca_weights_init(data_run) # using the pca initialization

    print('training')
    som.train_batch(data_run, n_init, verbose = True)

    end_time = datetime.now()
    print('Duration: {}'.format(end_time - start_time))


    #%% Retrieve SOM outputs

    def return_winner(point, matrix_result):
        """
        This function is to retrieve the number of the winning neuron for each point
        """

        win = som.winner(point) # this is to retrieve the BMU of each point
        r = win[0]
        c = win[1]
        neu = matrix_result[r,c]

        return neu

    neurs = np.arange(0,cols*rows).reshape((cols,rows))

    final_weights_normed = som.get_weights().copy() # this is to retrieve the final neurons
    winners = np.array([return_winner(data_run[i], neurs) for i in range(0,len(data_run))])

    if som_space:
        win_r = np.concatenate([winners,winners])
        #rs = data_nan.reshape((s_t*2,s_x,s_y))

    def denorm(neu_n, ar, ar_s):
        thing = neu_n*(np.nanmax(ar_s)-np.nanmin(ar_s))+np.nanmin(ar_s)
        neu = np.nanmean(ar)+np.nanstd(ar)*thing
        return neu

    neuron_u = np.nan*np.zeros((rows,cols,s_x,s_y))
    neuron_v = np.nan*np.zeros((rows,cols,s_x,s_y))

    for i in range(rows):
        for j in range(cols):

            neu = np.copy(data_nan[0,:])
            neu[~mask_nan] = final_weights_normed[i,j,:]
            neu_u_n = neu[0:s_x*s_y]
            neu_v_n = neu[s_x*s_y:2*s_x*s_y]

            neu_u = neu_u_n*store_std[0]+store_mean[0]
            neu_v = neu_v_n*store_std[1]+store_mean[1]

            neu_u[neu_u==0]=np.nan
            neu_v[neu_v==0]=np.nan

            neu_u_r = neu_u.reshape((s_x,s_y))
            neu_v_r = neu_v.reshape((s_x,s_y))

            neuron_u[i,j,:,:] = neu_u_r
            neuron_v[i,j,:,:] = neu_v_r
    np.savez(path_save_soms+name+'.npz',som=som,neuron_u=neuron_u,neuron_v=neuron_v,bmus=winners)
    print(name+' Done!')

## 1 A and 1 B

In [54]:

newcastle_u = newc['u'].values[~np.isnan(np.nanmean(newc['u'],axis=(1,2)))]
newcastle_v = newc['v'].values[~np.isnan(np.nanmean(newc['u'],axis=(1,2)))]

run_som_only_radar(newcastle_u, newcastle_v, rows=3, cols=3, name='newcastle_only')

coffs_u = coffs['u'].values[~np.isnan(np.nanmean(coffs['u'],axis=(1,2)))]
coffs_v = coffs['v'].values[~np.isnan(np.nanmean(coffs['u'],axis=(1,2)))]

run_som_only_radar(coffs_u, coffs_v, rows=3, cols=3, name='coffs_only')

/tmp/ipykernel_54685/3430005860.py:1: RuntimeWarning: Mean of empty slice
  newcastle_u = newc['u'].values[~np.isnan(np.nanmean(newc['u'],axis=(1,2)))]
/tmp/ipykernel_54685/3430005860.py:2: RuntimeWarning: Mean of empty slice
  newcastle_v = newc['v'].values[~np.isnan(np.nanmean(newc['u'],axis=(1,2)))]


You have chosen SOM spatial analysis
Your data shape is (813, 9522)
initializing


/home/nbourg/anaconda3/envs/my-conda-env/lib/python3.9/site-packages/xpysom/xpysom.py:693: ComplexWarning: Casting complex values to real discards the imaginary part
  self._weights[i, j] = c1*pc[pc_order[0]] + c2*pc[pc_order[1]]


training


 [ 162600 / 162600 ] 100% - 0:00:00 elapsed  - 0:00:00 left 
 quantization error: 16.66127586364746
Duration: 0:00:01.173166


newcastle_only Done!


/tmp/ipykernel_54685/3430005860.py:6: RuntimeWarning: Mean of empty slice
  coffs_u = coffs['u'].values[~np.isnan(np.nanmean(coffs['u'],axis=(1,2)))]
/tmp/ipykernel_54685/3430005860.py:7: RuntimeWarning: Mean of empty slice
  coffs_v = coffs['v'].values[~np.isnan(np.nanmean(coffs['u'],axis=(1,2)))]


You have chosen SOM spatial analysis
Your data shape is (544, 38076)
initializing
training
 [ 108800 / 108800 ] 100% - 0:00:01 elapsed  - 0:00:00 left 
 quantization error: 44.08597946166992
Duration: 0:02:59.627189
coffs_only Done!
